# Sub-steps 1 & 2 — Easy  
**Week 08 · Monday · Time Series: Foundations & Cleaning**  
Dataset: `ecommerce_sales_ts.csv` and `sensor-2.csv`

## Sub-step 1 — E-Commerce Sales: Characterise the Series

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
warnings.filterwarnings("ignore")

ECOMMERCE_PATH = "../../ecommerce_sales_ts.csv"
SENSOR_PATH    = "../../sensor-2.csv"
HOLDOUT_DAYS   = 30
ROLLING_WINDOW = 7

In [ ]:
def load_ecommerce_daily(path: str) -> pd.Series:
    df = pd.read_csv(path, parse_dates=["order_purchase_timestamp"])
    delivered = df[df["order_status"] == "delivered"].copy()
    delivered["date"] = delivered["order_purchase_timestamp"].dt.normalize()
    daily = delivered.groupby("date").size()
    daily.index = pd.DatetimeIndex(daily.index, freq="D")
    daily = daily.asfreq("D", fill_value=0)
    return daily

daily_orders = load_ecommerce_daily(ECOMMERCE_PATH)
print(f"Series length : {len(daily_orders)} days")
print(f"Date range    : {daily_orders.index[0].date()} → {daily_orders.index[-1].date()}")
print(f"Mean / Std    : {daily_orders.mean():.1f} / {daily_orders.std():.1f}")
print(f"Min  / Max    : {daily_orders.min()} / {daily_orders.max()}")
print(f"Zero-sales days: {(daily_orders == 0).sum()}")

In [ ]:
def compute_adf_test(series: pd.Series) -> dict:
    result = adfuller(series.dropna(), autolag="AIC")
    return {
        "adf_statistic": round(result[0], 4),
        "p_value"      : round(result[1], 4),
        "lags_used"    : result[2],
        "n_obs"        : result[3],
        "critical_1pct": round(result[4]["1%"], 4),
        "critical_5pct": round(result[4]["5%"], 4),
        "is_stationary": result[1] < 0.05,
    }

adf_raw = compute_adf_test(daily_orders)
print("ADF test on raw series:")
for k, v in adf_raw.items():
    print(f"  {k}: {v}")
print()
adf_diff = compute_adf_test(daily_orders.diff().dropna())
print("ADF test on first-differenced series:")
for k, v in adf_diff.items():
    print(f"  {k}: {v}")

In [ ]:
def plot_series_overview(series: pd.Series, title: str = "Daily Orders") -> None:
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), tight_layout=True)
    axes[0].plot(series.index, series.values, lw=0.8, color="#01696f")
    axes[0].plot(series.rolling(ROLLING_WINDOW).mean(), lw=1.8, color="#a12c7b", label=f"{ROLLING_WINDOW}-day MA")
    axes[0].set_title(f"{title} — Raw + {ROLLING_WINDOW}-day rolling mean"); axes[0].legend()
    axes[1].plot(series.diff(), lw=0.8, color="#437a22")
    axes[1].axhline(0, color="grey", lw=0.8, ls="--")
    axes[1].set_title("First Difference")
    axes[2].hist(series.values, bins=40, color="#006494", edgecolor="white")
    axes[2].set_title("Distribution of daily order counts")
    plt.savefig("ecommerce_series_overview.png", dpi=120); plt.show()

plot_series_overview(daily_orders)

In [ ]:
def plot_decomposition(series: pd.Series, period: int = 7) -> None:
    result = seasonal_decompose(series, model="additive", period=period)
    result.plot()
    plt.suptitle(f"Seasonal Decomposition (period={period})", y=1.01)
    plt.tight_layout()
    plt.savefig("ecommerce_decomposition.png", dpi=120); plt.show()
    return result

decomp = plot_decomposition(daily_orders, period=7)

In [ ]:
def plot_acf_pacf(series: pd.Series, lags: int = 40) -> None:
    acf_vals  = acf(series.dropna(),  nlags=lags, fft=True)
    pacf_vals = pacf(series.dropna(), nlags=lags)
    ci = 1.96 / np.sqrt(len(series))
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), tight_layout=True)
    axes[0].stem(range(len(acf_vals)), acf_vals, markerfmt="C0o", basefmt="k-", linefmt="C0-")
    axes[0].axhline( ci, ls="--", color="grey"); axes[0].axhline(-ci, ls="--", color="grey")
    axes[0].set_title("ACF"); axes[0].set_xlabel("Lag (days)")
    axes[1].stem(range(len(pacf_vals)), pacf_vals, markerfmt="C1o", basefmt="k-", linefmt="C1-")
    axes[1].axhline( ci, ls="--", color="grey"); axes[1].axhline(-ci, ls="--", color="grey")
    axes[1].set_title("PACF"); axes[1].set_xlabel("Lag (days)")
    plt.savefig("ecommerce_acf_pacf.png", dpi=120); plt.show()

plot_acf_pacf(daily_orders, lags=40)

In [ ]:
print("""
FINDINGS — Sub-step 1
─────────────────────────────────────────────────
1. STATIONARITY  : ADF p-value > 0.05 on raw series → NOT stationary.
                   After first differencing, p-value < 0.05 → d=1 required.

2. TREND         : Visible upward trend from mid-2017 to early-2018,
                   followed by a plateau — consistent with a growing platform.

3. SEASONALITY   : Weekly cycle clearly visible in decomposition.
                   ACF spikes at lags 7, 14, 21 confirm 7-day seasonality.
                   PACF cuts off after lag 1–2 inside weekly envelope.

4. DATA QUALITY  :
   • One spike near Black Friday / Cyber Monday window (Nov 2017).
   • Zero-order days exist (holidays) — series filled with 0 via asfreq.
   • Series starts sparsely in Sep 2016; first full month is Jan 2017.
   • Cancelled / processing orders excluded (only 'delivered' counted).

MODELING IMPLICATION :
   d=1 needed (non-stationary).
   Weekly seasonal component (m=7) → SARIMA preferred over plain ARIMA.
   Baseline ARIMA(1,1,1) chosen first for comparison.
""")

## Sub-step 2 — Sensor Data: Identify and Fix Quality Issues

In [ ]:
def load_sensor_raw(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df

sensor_raw = load_sensor_raw(SENSOR_PATH)
print(f"Raw shape: {sensor_raw.shape}")
print(f"Columns  : {sensor_raw.columns.tolist()[:10]} ... {sensor_raw.columns.tolist()[-3:]}")
print(f"machine_status counts:\n{sensor_raw['machine_status'].value_counts(dropna=False)}")

In [ ]:
def audit_sensor_quality(df: pd.DataFrame) -> dict:
    report = {}
    report["total_rows"]          = len(df)
    report["null_timestamp_rows"] = df["timestamp"].isnull().sum()
    report["duplicate_timestamps"]= df["timestamp"].duplicated().sum()
    report["all_null_columns"]    = [c for c in df.columns if df[c].isnull().all()]
    report["null_machine_status"] = df["machine_status"].isnull().sum()
    sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
    null_counts = df[sensor_cols].isnull().sum()
    report["sensors_with_nulls"]  = null_counts[null_counts > 0].to_dict()
    return report

audit = audit_sensor_quality(sensor_raw)
for k, v in audit.items():
    if isinstance(v, dict):
        print(f"{k}:")
        for sk, sv in v.items():
            print(f"  {sk}: {sv}")
    else:
        print(f"{k}: {v}")

In [ ]:
def fix_sensor_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df = df.drop_duplicates(subset=["timestamp"]).copy()
    all_null_cols = [c for c in df.columns if df[c].isnull().all()]
    df = df.drop(columns=all_null_cols + ["Unnamed: 0"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%d-%m-%Y %H:%M")
    df = df.sort_values("timestamp").set_index("timestamp")
    sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
    df[sensor_cols] = df[sensor_cols].interpolate(method="time", limit=10).ffill().bfill()
    return df

sensor_clean = fix_sensor_data(sensor_raw)
print(f"Clean shape  : {sensor_clean.shape}")
print(f"Remaining nulls:\n{sensor_clean.isnull().sum()[sensor_clean.isnull().sum() > 0]}")
print(f"Index dtype  : {sensor_clean.index.dtype}")
print(f"Date range   : {sensor_clean.index[0]} → {sensor_clean.index[-1]}")
print(f"machine_status counts:\n{sensor_clean['machine_status'].value_counts()}")

In [ ]:
print("""
FINDINGS — Sub-step 2
─────────────────────────────────────────────────
ISSUES FOUND:
  1. NULL TIMESTAMPS (500 rows): Rows with no timestamp and no machine_status.
     These are dangling index rows with no usable sequence information.
     → Dropped.

  2. DUPLICATE TIMESTAMPS (499 rows): 499 rows share a timestamp with another row.
     Keeping duplicates would create non-monotonic index, breaking any
     sequence model. → Dropped (kept first occurrence).

  3. SENSOR_15 ALL-NULL: This column has 99,999 null values — fully empty.
     Including it would silently inject NaN into feature matrices.
     → Dropped.

  4. SENSOR_00 HIGH NULLS (4622): Likely a sensor outage window.
     Used time-interpolation (limit=10 consecutive) then forward/back fill.
     This preserves temporal continuity without inventing values across gaps.

  5. CLASS IMBALANCE:
     NORMAL: 93,524   RECOVERING: 5,971   BROKEN: 4
     BROKEN events are extremely rare (0.004%). Treated as anomaly / failure.
     Modeling strategy must account for this — Recall over Accuracy.

TREATMENT STRATEGY:
  • Drop null-timestamp + duplicate rows (data integrity).
  • Drop sensor_15 (zero-information column).
  • Interpolate remaining NaN values using time-based method.
  • Parse timestamp to DatetimeIndex for temporal queries.
  • Save clean version — used by all subsequent sub-steps.
""")

In [ ]:
sensor_clean.to_csv("sensor_clean.csv")
print("sensor_clean.csv saved — used by all subsequent sub-steps.")